# Build Measured Intrinsic — Batch Driver (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-05-14
**Last Modified:** 2026-05-14
**Status:** Draft
**Keywords:** AOS, intrinsic, batch, papermill, parameter scan

## Description

Runs `build_measured_intrinsic.ipynb` once per parameter set via
**papermill**, so a scan over elevation / rotator ranges, filter
bands, and `n_keep` can be launched from a single list instead of
hand-editing the params cell and re-running.

Each run:
* gets its parameters injected into the tagged `parameters` cell of
  `build_measured_intrinsic.ipynb`,
* writes its normal PDFs + parquet into its own self-describing
  `output_dir`
  (`output/build_measured_intrinsic/{binning}_nkeep_{n_keep}_{coord_sys}_…/`),
* leaves an executed notebook copy under
  `output/build_measured_intrinsic/_runs/{tag}.ipynb` for the record.

A run **manifest** (`_runs/manifest.parquet` + `.csv`) collects one row
per run: the parameters, the resolved `output_dir`, the executed-notebook
path, and pass/fail status.

**Output:** executed notebooks + manifest in
`output/build_measured_intrinsic/_runs/`; per-run analysis products in
each run's own `output_dir`.

**Based on:** `build_measured_intrinsic.ipynb` (the worker notebook).


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-14 | Aaron Roodman | Initial version. |


## Table of Contents

1. [Parameters & Parameter Sets](#params)
2. [Setup](#setup)
3. [Run the batch](#run)
4. [Manifest summary](#manifest)


<a id='params'></a>
## 1. Parameters & Parameter Sets

In [1]:
# ============================================================
# Batch parameters
# ============================================================
from pathlib import Path

# Worker notebook to execute once per parameter set.
worker_notebook = 'build_measured_intrinsic.ipynb'

# Where executed-notebook copies + the run manifest are written.
runs_dir = Path('output/build_measured_intrinsic/_runs')

# Per-run overrides.  Each dict is injected into the worker's tagged
# `parameters` cell; any key NOT listed keeps the worker's default.
# Only list the knobs you are scanning — the rest (n_iter, n_bins,
# height-map, etc.) come from the worker's params cell.
#
# Tip: keep `binning` and `coord_sys` explicit per set so the
# auto-derived output_dir name is unambiguous.
PARAM_SETS = [

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=-65.0, rot_max_deg=65.0,
         allowed_bands=['r', 'i', 'z'], n_keep=25),
    
    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=-3.0, rot_max_deg=3.0,
         allowed_bands=['r', 'i', 'z'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=55.0, rot_max_deg=65.0,
         allowed_bands=['r', 'i', 'z'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=-65.0, rot_max_deg=-55.0,
         allowed_bands=['r', 'i', 'z'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=-20.0, rot_max_deg=-10.0,
         allowed_bands=['r', 'i', 'z'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=10.0, rot_max_deg=20.0,
         allowed_bands=['r', 'i', 'z'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=55.0, alt_max_deg=75.0,
         rot_min_deg=-10.0, rot_max_deg=10.0,
         allowed_bands=['u'], n_keep=34),

#    dict(binning='bin_2x', coord_sys='OCS',
#         alt_min_deg=55.0, alt_max_deg=75.0,
#         rot_min_deg=-10.0, rot_max_deg=10.0,
#         allowed_bands=['y'], n_keep=34),

    dict(binning='bin_2x', coord_sys='OCS',
         alt_min_deg=35.0, alt_max_deg=55.0,
         rot_min_deg=-65.0, rot_max_deg=65.0,
         allowed_bands=['r', 'i', 'z'], n_keep=25),
]

# ----- Subset selection -----
# Which PARAM_SETS to run this time.  Options:
#   None            -> run every set
#   [0, 3, 5]       -> run only those indices (see the printed list
#                      from the setup cell)
#   'bin_2x_...riz' -> a run tag (or substring) to match by name
#   ['...u', '...y']-> a list of tags / substrings
run_indices = [6]

# When True, skip any set whose executed notebook already exists in
# runs_dir (handy for resuming a partially-finished batch).
skip_existing = False

# Stop the whole batch on the first failure, or keep going and record
# failures in the manifest?
stop_on_error = False

# Kernel name papermill uses to execute the worker.  None lets
# papermill pick the worker's saved kernelspec.
kernel_name = None


<a id='setup'></a>
## 2. Setup

In [2]:
import sys
import json
import traceback

import numpy as np
import pandas as pd

try:
    import papermill as pm
    _pm_ok = True
except Exception as _e:
    print(f'papermill import failed: {type(_e).__name__}: {_e}')
    print('Install with:  pip install papermill')
    _pm_ok = False

sys.path.insert(0, 'code')
# Reuse the worker's output-dir naming so the manifest can record the
# exact directory each run wrote to (no need to re-execute logic).
from run_pipeline import (load_runs, load_param_sets, resolve_run,
                          donut_path)   # noqa: F401  (kept for parity)


def derive_output_dir(ps):
    """Reproduce build_measured_intrinsic.output_dir_default for a param set.

    Mirrors the worker's naming:
      output/build_measured_intrinsic/{binning}_nkeep_{n_keep}_{coord_sys}
      [ _day_.._.. ][ _seq_.._.. ][ _alt_.._.. ][ _rot_.._.. ]
    Only the keys present in `ps` (plus defaults) matter; we default to
    the worker's params-cell values for anything unset.
    """
    g = dict(binning='bin_2x', coord_sys='OCS', n_keep=25,
             day_obs_min=None, day_obs_max=None,
             seq_num_min=None, seq_num_max=None,
             alt_min_deg=55.0, alt_max_deg=75.0,
             rot_min_deg=-3.0, rot_max_deg=3.0,
             allowed_bands=None)
    g.update(ps)
    _band = ''.join(str(b) for b in g['allowed_bands']) \
        if g.get('allowed_bands') else 'allband'
    parts = [f"{g['binning']}", f"nkeep_{int(g['n_keep'])}",
             f"{g['coord_sys']}", _band]
    if g['day_obs_min'] or g['day_obs_max']:
        parts.append(f"day_{g['day_obs_min'] or ''}_{g['day_obs_max'] or ''}")
    if g['seq_num_min'] or g['seq_num_max']:
        parts.append(f"seq_{g['seq_num_min'] or ''}_{g['seq_num_max'] or ''}")
    if g['alt_min_deg'] is not None or g['alt_max_deg'] is not None:
        parts.append(f"alt_{g['alt_min_deg'] or ''}_{g['alt_max_deg'] or ''}")
    if g['rot_min_deg'] is not None or g['rot_max_deg'] is not None:
        parts.append(f"rot_{g['rot_min_deg'] or ''}_{g['rot_max_deg'] or ''}")
    return Path('output') / 'build_measured_intrinsic' / '_'.join(parts)


def run_tag(ps):
    """Short filesystem-safe tag for a param set (used for the run nb name)."""
    g = dict(binning='bin_2x', coord_sys='OCS', n_keep=25,
             alt_min_deg=55.0, alt_max_deg=75.0,
             rot_min_deg=-3.0, rot_max_deg=3.0,
             allowed_bands=None)
    g.update(ps)
    bands = ''.join(g['allowed_bands']) if g.get('allowed_bands') else 'allband'
    return (f"{g['binning']}_nkeep{int(g['n_keep'])}_{g['coord_sys']}_"
            f"alt{g['alt_min_deg']:g}_{g['alt_max_deg']:g}_"
            f"rot{g['rot_min_deg']:g}_{g['rot_max_deg']:g}_{bands}")


print(f'{len(PARAM_SETS)} parameter set(s) queued.')
for i, ps in enumerate(PARAM_SETS):
    print(f'  [{i}] {run_tag(ps)}')

def select_param_sets(param_sets, run_indices):
    """Resolve `run_indices` into a list of (index, param_set) to run.

    `run_indices` may be None (all), an int, a list of ints, a tag /
    substring string, or a list of tag / substring strings.
    """
    if run_indices is None:
        return list(enumerate(param_sets))
    if isinstance(run_indices, (int, str)):
        run_indices = [run_indices]
    selected = []
    for sel in run_indices:
        if isinstance(sel, int):
            selected.append((sel, param_sets[sel]))
        else:  # string: match against run_tag
            hits = [(i, ps) for i, ps in enumerate(param_sets)
                    if sel in run_tag(ps)]
            if not hits:
                raise ValueError(f'run_indices entry {sel!r} matched no '
                                 f'parameter set')
            selected.extend(hits)
    # De-dup while preserving order.
    seen, out = set(), []
    for i, ps in selected:
        if i not in seen:
            seen.add(i); out.append((i, ps))
    return out



8 parameter set(s) queued.
  [0] bin_2x_nkeep34_OCS_alt55_75_rot-3_3_riz
  [1] bin_2x_nkeep34_OCS_alt55_75_rot55_65_riz
  [2] bin_2x_nkeep34_OCS_alt55_75_rot-65_-55_riz
  [3] bin_2x_nkeep34_OCS_alt55_75_rot-20_-10_riz
  [4] bin_2x_nkeep34_OCS_alt55_75_rot10_20_riz
  [5] bin_2x_nkeep34_OCS_alt55_75_rot-10_10_u
  [6] bin_2x_nkeep34_OCS_alt55_75_rot-10_10_y
  [7] bin_2x_nkeep25_OCS_alt30_50_rot-3_3_riz


<a id='run'></a>
## 3. Run the batch

Each param set is injected into the worker notebook's tagged
`parameters` cell and executed end-to-end.  Failures are
recorded (unless `stop_on_error=True`) so one bad config
doesn't sink the whole scan.

**To rerun a single set** (or a subset) set `run_indices` in the
parameters cell — by index (`run_indices = [2]`), by run-tag
substring (`run_indices = '_u'` to grab the u-band run), or a
mix.  `skip_existing = True` skips sets whose executed notebook
already exists, to resume a partial batch.


In [3]:
assert _pm_ok, 'papermill is required for the batch run (pip install papermill)'
runs_dir.mkdir(parents=True, exist_ok=True)

selected = select_param_sets(PARAM_SETS, run_indices)
print(f'Running {len(selected)} of {len(PARAM_SETS)} parameter set(s): '
      f'{[i for i, _ in selected]}')

manifest_rows = []
for i, ps in selected:
    tag = run_tag(ps)
    out_nb = runs_dir / f'{tag}.ipynb'
    od = derive_output_dir(ps)
    if skip_existing and out_nb.exists():
        print(f'\n=== [{i}] {tag} ===  (skip_existing: already done)')
        continue
    print(f'\n=== [{i}] {tag} ===')
    print(f'    params:     {ps}')
    print(f'    output_dir: {od}')
    print(f'    run nb:     {out_nb}')

    row = dict(ps)
    row['tag'] = tag
    row['output_dir'] = str(od)
    row['executed_notebook'] = str(out_nb)
    try:
        pm.execute_notebook(
            worker_notebook, str(out_nb),
            parameters=ps,
            kernel_name=kernel_name,
            progress_bar=False,   # avoid the ipywidget render error in headless runs
        )
        row['status'] = 'ok'
        row['error'] = ''
        print(f'    -> OK')
    except Exception as e:
        row['status'] = 'failed'
        row['error'] = f'{type(e).__name__}: {e}'
        print(f'    -> FAILED: {row["error"]}')
        traceback.print_exc()
        if stop_on_error:
            manifest_rows.append(row)
            raise
    manifest_rows.append(row)

print(f'\nBatch complete: '
      f'{sum(r["status"] == "ok" for r in manifest_rows)} ok, '
      f'{sum(r["status"] == "failed" for r in manifest_rows)} failed.')


Running 1 of 8 parameter set(s): [6]

=== [6] bin_2x_nkeep34_OCS_alt55_75_rot-10_10_y ===
    params:     {'binning': 'bin_2x', 'coord_sys': 'OCS', 'alt_min_deg': 55.0, 'alt_max_deg': 75.0, 'rot_min_deg': -10.0, 'rot_max_deg': 10.0, 'allowed_bands': ['y'], 'n_keep': 34}
    output_dir: output/build_measured_intrinsic/bin_2x_nkeep_34_OCS_y_alt_55.0_75.0_rot_-10.0_10.0
    run nb:     output/build_measured_intrinsic/_runs/bin_2x_nkeep34_OCS_alt55_75_rot-10_10_y.ipynb


    -> FAILED: PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [5]":
---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[5], line 38
     34 donut_df = filter_donuts_by_visits(donut_full, visits_kept)
     35 print(f'Donuts after filters: {len(donut_df)}/{len(donut_full)}')
     36 
     37 # Derive Noll indices from the data
---> 38 nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
     39 noll_arr = (np.array(visits_kept['nollIndices'][0])
     40             if 'nollIndices' in visits_kept.colnames else None)
     41 iZs, iZidx = derive_noll_indices(nZk, noll_arr)

File /opt/lsst/software/stack/conda/envs/lsst-scipipe-13.0.0-exact/lib/python3.13/site-packages/numpy/_core/shape_base.py:456, in stack(arrays, axis, out, dtype, casting)
    454 arrays = [asanyarray(arr) for arr in arrays]


Traceback (most recent call last):
  File "/tmp/ipykernel_689/1603371299.py", line 26, in <module>
    pm.execute_notebook(
    ~~~~~~~~~~~~~~~~~~~^
        worker_notebook, str(out_nb),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
        progress_bar=False,   # avoid the ipywidget render error in headless runs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/opt/lsst/software/stack/conda/envs/lsst-scipipe-13.0.0-exact/lib/python3.13/site-packages/papermill/execute.py", line 131, in execute_notebook
    raise_for_execution_errors(nb, output_path)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/lsst/software/stack/conda/envs/lsst-scipipe-13.0.0-exact/lib/python3.13/site-packages/papermill/execute.py", line 251, in raise_for_execution_errors
    raise error
papermill.exceptions.PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [

<a id='manifest'></a>
## 4. Manifest summary

In [4]:
# Build a manifest of all runs in this batch and persist it.
manifest = pd.DataFrame(manifest_rows)
# Render list-valued columns (allowed_bands) as strings for parquet/csv.
for col in manifest.columns:
    if manifest[col].apply(lambda v: isinstance(v, (list, tuple))).any():
        manifest[col] = manifest[col].apply(
            lambda v: ','.join(map(str, v)) if isinstance(v, (list, tuple))
            else v)

manifest_parquet = runs_dir / 'manifest.parquet'
manifest_csv     = runs_dir / 'manifest.csv'
manifest.to_parquet(manifest_parquet)
manifest.to_csv(manifest_csv, index=False)
print(f'Wrote manifest: {manifest_parquet}')
print(f'                {manifest_csv}')

with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.width', 200):
    display(manifest)


Wrote manifest: output/build_measured_intrinsic/_runs/manifest.parquet
                output/build_measured_intrinsic/_runs/manifest.csv


,binning,coord_sys,alt_min_deg,alt_max_deg,rot_min_deg,rot_max_deg,allowed_bands,n_keep,tag,output_dir,executed_notebook,status,error
0,bin_2x,OCS,55.0,75.0,-10.0,10.0,y,34,bin_2x_nkeep34_OCS_alt55_75_rot-10_10_y,output/build_measured_intrinsic/bin_2x_nkeep_3...,output/build_measured_intrinsic/_runs/bin_2x_n...,failed,PapermillExecutionError: \n-------------------...
